# Crop Type Labeling with Cecil + WherobotsDB

This notebook demonstrates how to combine **Cecil** satellite data acquisition with **WherobotsDB** spatial analytics to label agricultural field boundaries with their crop type.

**Workflow:**
1. Use **WKLS** to get the boundary of Haskell County, Kansas
2. Use the **Cecil API** to acquire USDA Cropland Data Layer (CDL) raster data for the AOI
3. Discover the GeoTIFF file paths on S3
4. Load the CDL rasters into **WherobotsDB** as out-of-database rasters using `RS_FromPath`
5. Load **Fields of the World (FTW)** field boundary polygons from the **Wherobots Open Data Catalog**
6. Run **`RS_ZonalStats`** to compute the dominant crop class for each field polygon
7. Map CDL class codes to human-readable crop names

**Prerequisites:**
- A Wherobots Cloud notebook environment
- A Cecil API key (set as `CECIL_API_KEY` environment variable)
- An active Cecil subscription with access to the USDA CDL dataset

## 1. Setup & Dependencies

In [ ]:
%pip install cecil

In [ ]:
import json
import os

import boto3
import cecil
import wkls
from cecil.models.subscription import SubscriptionTIFF
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. Define the Area of Interest with WKLS

We use [WKLS](https://github.com/wherobots/wkls) (Well-Known Locations) to get the precise
boundary of Haskell County, Kansas. WKLS provides administrative boundaries from the
Overture Maps Foundation, no manual coordinate entry needed.

In [ ]:
# Get Haskell County KS boundary in GeoJSON and WKT formats
haskell_geojson_str = wkls.us.ks.haskellcounty.geojson()
haskell_wkt = wkls.us.ks.haskellcounty.wkt()

# Parse GeoJSON string to dict (needed for Cecil API)
haskell_geojson = json.loads(haskell_geojson_str)

print(f"Geometry type: {haskell_geojson['type']}")
print(f"WKT preview: {haskell_wkt[:120]}...")

## 3. Acquire CDL Raster Data from Cecil

The [USDA Cropland Data Layer (CDL)](https://www.nass.usda.gov/Research_and_Science/Cropland/SARS1a.php)
is a 10m resolution raster covering CONUS with 135+ crop classes. It is freely available
through Cecil.

### 3a. Create AOI and Subscribe to CDL

Cecil requires an AOI and a subscription to deliver data. The subscription is asynchronous, it may take some time for the data to be processed and available.

In [ ]:
import getpass

# Prompts at runtime and masks input, the key is never stored in the notebook
os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API Key: ")

In [ ]:
cecil_client = cecil.Client()

# Create an AOI using the WKLS geometry
aoi = cecil_client.create_aoi(
    external_ref="Haskell County, KS",
    geometry=haskell_geojson,
)

# USDA CDL 10m dataset ID on Cecil
CDL_DATASET_ID = "84f5bd14-83fc-4b45-8a65-ebe31eea4da1"

# USDA CDL 30m dataset ID on Cecil
CDL_30m_DATASET_ID = "86bb24b3-c6c8-4a6c-8641-ae899852e3e3"

subscription = cecil_client.create_subscription(
    external_ref="CDL 30m - Haskell County KS",
    aoi_id=aoi.id,
    dataset_id=CDL_30m_DATASET_ID,
)

print(f"AOI ID: {aoi.id}")
print(f"Area: {aoi.hectares:.0f} hectares")
print(f"Subscription ID: {subscription.id}")

> **Note:** If you already have a subscription, you can skip the cell above and set the
> subscription ID directly:
> ```python
> subscription_id = "your-existing-subscription-id"
> ```

### 3b. Get Temporary Credentials and Discover GeoTIFF Files

Cecil provides temporary AWS STS credentials scoped to the subscription's S3 prefix.
We use these to discover the available GeoTIFF files and build S3 paths for WherobotsDB.

The bucket name, prefix, and file paths are stable per subscription, only the
credentials need to be refreshed. Once you know the paths, you can hardcode them
and only re-run the credential fetch.

> **Note: new subscriptions are fulfilled asynchronously.** Right after
> `create_subscription()`, Cecil returns the bucket, prefix, and credentials
> immediately, but the GeoTIFFs may not be written to the prefix yet. If the
> listing cell below prints `Found 0 GeoTIFF file(s)`, the data is most likely
> still processing (not a code or permissions error). Wait a moment and
> **re-run the listing cell**, the credentials stay valid (~12 h), so you don't
> need to re-fetch them. It will return files once processing completes.

In [ ]:
subscription_id = subscription.id  # or set manually if using an existing subscription

# Call Cecil API to get S3 credentials and file metadata
res = SubscriptionTIFF(
    **cecil_client._get(url=f"/v0/subscriptions/{subscription_id}/files/tiff")
)

print(f"Dataset:  {res.dataset_name}")
print(f"Bucket:   {res.bucket.name}")
print(f"Prefix:   {res.bucket.prefix}")
print(f"Region:   {res.credentials.region}")
print(f"Expires:  {res.credentials.expiration}")
print(f"\nBand metadata:")
for filename, file_info in res.file_mapping.items():
    for band in file_info.bands:
        print(f"  {filename}: band {band.number} = '{band.name}' ({band.dtype}, nodata={band.nodata})")

In [ ]:
# List GeoTIFF files under the subscription prefix
session = boto3.session.Session(
    aws_access_key_id=res.credentials.access_key_id,
    aws_secret_access_key=res.credentials.secret_access_key,
    aws_session_token=res.credentials.session_token,
    region_name=res.credentials.region,
)

s3 = session.client("s3")
paginator = s3.get_paginator("list_objects_v2")

tiff_keys = []
for page in paginator.paginate(Bucket=res.bucket.name, Prefix=res.bucket.prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].lower().endswith((".tif", ".tiff")):
            tiff_keys.append(obj["Key"])

print(f"Found {len(tiff_keys)} GeoTIFF file(s):")
for key in tiff_keys:
    print(f"  s3a://{res.bucket.name}/{key}")

## 4. Load CDL Rasters into WherobotsDB

We use `RS_FromPath` with inline Hadoop filesystem parameters to pass Cecil's temporary
STS credentials directly. This creates out-of-database rasters that fetch pixel data
on-demand via HTTP range requests, efficient for large COGs.

In [ ]:
# Build the Hadoop S3A credential string for RS_FromPath
cred_params = (
    f"fs.s3a.access.key={res.credentials.access_key_id};"
    f"fs.s3a.secret.key={res.credentials.secret_access_key};"
    f"fs.s3a.session.token={res.credentials.session_token}"
)

# Build S3A paths (note: s3a:// protocol, not s3://)
s3a_paths = [f"s3a://{res.bucket.name}/{key}" for key in tiff_keys]

# Create a DataFrame of S3 paths
paths_df = sedona.createDataFrame([(p,) for p in s3a_paths], ["path"])

# Load as out-db rasters with Cecil credentials
cdl_raster_df = paths_df.withColumn(
    "rast",
    F.expr(f"RS_FromPath(path, '{cred_params}')")
)

cdl_raster_df.cache()
cdl_raster_df.createOrReplaceTempView("cdl_rasters")

print(f"Loaded {cdl_raster_df.count()} CDL raster(s)")
cdl_raster_df.selectExpr(
    "path",
    "RS_Width(rast) AS width",
    "RS_Height(rast) AS height",
    "RS_SRID(rast) AS srid",
    "RS_NumBands(rast) AS num_bands",
).show(truncate=60)

## 5. Load FTW Field Boundaries from the Wherobots Open Data Catalog

Load field boundary polygons from the **Fields of the World (FTW)** global dataset, hosted
in the **Wherobots Open Data Catalog** as a managed Iceberg table, no external bucket or
credentials required, and spatial predicates are pushed down automatically.

**Table:** `wherobots_open_data.rasterflow_output_samples.fields_of_the_world_vector_global`

**Schema:**
- `geometry`, field boundary polygon
- `time`, timestamp of the prediction
- `layer`, one of: `field`, `field_boundaries`, `non_field_background`
- `bbox`, struct with `xmin`, `ymin`, `xmax`, `ymax`

In [ ]:
FTW_TABLE = "wherobots_open_data.rasterflow_output_samples.fields_of_the_world_vector_global"

ftw_all = sedona.table(FTW_TABLE)

print("FTW table schema:")
ftw_all.printSchema()

In [ ]:
# Filter to "field" polygons and intersect with the Haskell County boundary
ftw_fields = ftw_all.filter(
    (F.col("layer") == "field") &
    F.expr(f"ST_Intersects(geometry, ST_GeomFromWKT('{haskell_wkt}'))")
)

ftw_fields.cache()
ftw_fields.createOrReplaceTempView("ftw_fields")

field_count = ftw_fields.count()
print(f"FTW fields in Haskell County: {field_count}")
ftw_fields.show(5, truncate=60)

In [ ]:
# Optional: Save filtered FTW fields to the WherobotsDB catalog for faster reuse
# ftw_fields.writeTo("org_catalog.ftw_db.ftw_haskell_county").create()

print("Uncomment the line above to save filtered FTW fields to the catalog.")

## 6. Zonal Stats, Crop Type per Field

Use `RS_ZonalStats` with `mode` to compute the most frequently occurring CDL pixel value
within each field polygon. This gives us the dominant crop class for each field.

Key details:
- CDL is in **EPSG:5070** (Albers Equal Area), `RS_ZonalStats` automatically transforms
  the EPSG:4326 field geometries to match
- `mode` returns the most common pixel value; ties go to the largest value
- `lenient=true` (default) returns `null` for non-intersecting raster/geometry pairs

In [ ]:
# Cross join field polygons with CDL rasters and compute zonal mode
crop_labels_df = sedona.sql("""
    SELECT
        f.*,
        CAST(RS_ZonalStats(c.rast, f.geometry, 1, 'mode', true) AS INT) AS cdl_class
    FROM ftw_fields f
    CROSS JOIN cdl_rasters c
""")

# Drop rows where the raster and polygon did not intersect
crop_labels_df = crop_labels_df.filter(F.col("cdl_class").isNotNull())

crop_labels_df.cache()
crop_labels_df.createOrReplaceTempView("crop_labels")

print(f"Fields with crop labels: {crop_labels_df.count()}")
crop_labels_df.select("geometry", "cdl_class").show(10, truncate=60)

## 7. Map CDL Class Codes to Crop Names

The CDL uses integer codes for crop types. Below is a lookup table for the most common
classes you'll find in western Kansas (Haskell County is in the wheat/sorghum/corn belt).

Full CDL class definitions: https://www.nass.usda.gov/Research_and_Science/Cropland/metadata/meta.php

In [ ]:
# USDA CDL class code to crop name mapping (common classes)
CDL_CLASSES = {
    1: "Corn",
    2: "Cotton",
    3: "Rice",
    4: "Sorghum",
    5: "Soybeans",
    6: "Sunflower",
    10: "Peanuts",
    11: "Tobacco",
    12: "Sweet Corn",
    13: "Pop/Orn Corn",
    14: "Mint",
    21: "Barley",
    22: "Durum Wheat",
    23: "Spring Wheat",
    24: "Winter Wheat",
    25: "Other Small Grains",
    26: "Dbl Crop WinWht/Soybeans",
    27: "Rye",
    28: "Oats",
    29: "Millet",
    30: "Speltz",
    31: "Canola",
    32: "Flaxseed",
    33: "Safflower",
    34: "Rape Seed",
    35: "Mustard",
    36: "Alfalfa",
    37: "Other Hay/Non Alfalfa",
    38: "Camelina",
    39: "Buckwheat",
    41: "Sugarbeets",
    42: "Dry Beans",
    43: "Potatoes",
    44: "Other Crops",
    45: "Sugarcane",
    46: "Sweet Potatoes",
    51: "Chick Peas",
    52: "Lentils",
    53: "Peas",
    54: "Tomatoes",
    55: "Caneberries",
    56: "Hops",
    57: "Herbs",
    58: "Clover/Wildflowers",
    59: "Sod/Grass Seed",
    60: "Switchgrass",
    61: "Fallow/Idle Cropland",
    63: "Forest",
    64: "Shrubland",
    65: "Barren",
    66: "Cherries",
    67: "Peaches",
    68: "Apples",
    69: "Grapes",
    70: "Christmas Trees",
    71: "Other Tree Crops",
    72: "Citrus",
    74: "Pecans",
    75: "Almonds",
    76: "Walnuts",
    77: "Pears",
    81: "Clouds/No Data",
    82: "Developed",
    83: "Water",
    87: "Wetlands",
    88: "Nonag/Undefined",
    92: "Aquaculture",
    111: "Open Water",
    112: "Perennial Ice/Snow",
    121: "Developed/Open Space",
    122: "Developed/Low Intensity",
    123: "Developed/Med Intensity",
    124: "Developed/High Intensity",
    131: "Barren",
    141: "Deciduous Forest",
    142: "Evergreen Forest",
    143: "Mixed Forest",
    152: "Shrubland",
    176: "Grassland/Pasture",
    190: "Woody Wetlands",
    195: "Herbaceous Wetlands",
    204: "Pistachios",
    205: "Triticale",
    206: "Carrots",
    207: "Asparagus",
    208: "Garlic",
    209: "Cantaloupes",
    210: "Prunes",
    211: "Olives",
    212: "Oranges",
    213: "Honeydew Melons",
    214: "Broccoli",
    216: "Peppers",
    217: "Pomegranates",
    218: "Nectarines",
    219: "Greens",
    220: "Plums",
    221: "Strawberries",
    222: "Squash",
    223: "Apricots",
    224: "Vetch",
    225: "Dbl Crop WinWht/Corn",
    226: "Dbl Crop Oats/Corn",
    227: "Lettuce",
    229: "Pumpkins",
    230: "Dbl Crop Lettuce/Durum Wht",
    231: "Dbl Crop Lettuce/Cantaloupe",
    232: "Dbl Crop Lettuce/Cotton",
    233: "Dbl Crop Lettuce/Barley",
    234: "Dbl Crop Durum Wht/Sorghum",
    235: "Dbl Crop Barley/Sorghum",
    236: "Dbl Crop WinWht/Sorghum",
    237: "Dbl Crop Barley/Corn",
    238: "Dbl Crop WinWht/Cotton",
    239: "Dbl Crop Soybeans/Cotton",
    240: "Dbl Crop Soybeans/Oats",
    241: "Dbl Crop Corn/Soybeans",
    242: "Blueberries",
    243: "Cabbage",
    244: "Cauliflower",
    245: "Celery",
    246: "Radishes",
    247: "Turnips",
    248: "Eggplants",
    249: "Gourds",
    250: "Cranberries",
    254: "Dbl Crop Barley/Soybeans",
}

# Create a Spark lookup DataFrame
cdl_lookup_rows = [(code, name) for code, name in CDL_CLASSES.items()]
cdl_lookup_df = sedona.createDataFrame(cdl_lookup_rows, ["cdl_class", "crop_name"])
cdl_lookup_df.createOrReplaceTempView("cdl_lookup")

print(f"CDL lookup table: {len(CDL_CLASSES)} classes")

In [ ]:
# Join crop labels with the lookup table
enriched_fields = sedona.sql("""
    SELECT
        cl.*,
        COALESCE(lu.crop_name, CONCAT('Unknown (', cl.cdl_class, ')')) AS crop_name
    FROM crop_labels cl
    LEFT JOIN cdl_lookup lu ON cl.cdl_class = lu.cdl_class
""")

enriched_fields.cache()
enriched_fields.createOrReplaceTempView("enriched_fields")

enriched_fields.select("geometry", "cdl_class", "crop_name").show(20, truncate=60)

In [ ]:
# Crop distribution summary
crop_summary = sedona.sql("""
    SELECT
        crop_name,
        cdl_class,
        COUNT(*) AS field_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM enriched_fields
    GROUP BY crop_name, cdl_class
    ORDER BY field_count DESC
""")

print("Crop type distribution in Haskell County fields:")
crop_summary.show(20, truncate=40)

## 8. Visualize Results

Display the enriched field polygons on an interactive map using SedonaKepler.

In [ ]:
from sedona.spark.maps.SedonaKepler import SedonaKepler

# Prepare a display DataFrame with geometry and crop name
display_df = enriched_fields.select("geometry", "crop_name", "cdl_class")

map_viz = SedonaKepler.create_map(df=display_df, name="Crop Types")
map_viz

## 9 Generate PMTiles (Optional)

To improve visualization performance for a large number of geometries, generate PMTiles from the results.

Note: this may require a larger Wherobots notebook instance to complete in a timely manner.

In [ ]:
# from wherobots import vtiles

# fields_df = ftw_all.where("layer = 'field'").select("geometry").withColumn("layer", F.lit("fields"))
# boundaries_df = (
#     ftw_all.where("layer = 'field_boundaries'")
#     .select("geometry")
#     .withColumn("layer", F.lit("boundaries"))
# )

# tile_features_df = fields_df.unionByName(boundaries_df)

# user_s3_path = os.getenv("USER_S3_PATH")

# full_tiles_path = user_s3_path.rstrip("/") + "/haskell_county_crop_labels.pmtiles"
# vtiles.generate_pmtiles(tile_features_df, full_tiles_path)
# vtiles.show_pmtiles(full_tiles_path)

print("Uncomment the option above to generate PMTiles.")

## 10. Export Results (Optional)

Save the enriched field boundaries to the WherobotsDB catalog or to GeoParquet on S3.

In [ ]:
# Option A: Save as a Havasu table in WherobotsDB
# enriched_fields.writeTo("wherobots_open_data.default.haskell_county_crop_labels").create()

# Option B: Save as GeoParquet to managed storage
# enriched_fields.write.format("geoparquet").mode("overwrite").save(
#     "s3://wherobots-managed-storage/output/haskell_county_crop_labels"
# )

print("Uncomment the export option above to save results.")